<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-255-Data-Mining/blob/main/Data_Warehousing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



__Problem 1__      (20pt)
Suppose that a data warehouse consists of the three dimensions time, doctor, and patient, and the two measures count and charge, where charge is the fee that a doctor charges a patient for a visit.

1. Enumerate three classes of schemas that are popularly used for modeling data warehouses.
2. Draw a schema diagram for the above data warehouse using one of the schema classes listed in (a).
3. Starting with the base cuboid [day, doctor, patient], what specific OLAP operations should be performed in order to list the total fee collected by each doctor in 2024?
4. To obtain the same list, write an SQL query assuming the data is stored in a relational database with the schema fee (day, month, year, doctor, hospital, patient, count, charge).

###ANSWER:

a)
The three classes of schemas are:
1.  **Star Schema:** A single central fact table connected to multiple surrounding denormalized dimension tables.
2.  **Snowflake Schema:** An extension of the star schema where dimension tables are normalized, splitting data into additional tables to reduce redundancy.
3.  **Fact Constellation Schema (Galaxy Schema):** A schema containing multiple fact tables that share dimension tables.

### b) Schema Diagram


**Fact Table: Fee_Fact**
* Time_ID (Primary Key, Foreign Key)
* Doctor_ID (Primary Key, Foreign Key)
* Patient_ID (Primary Key, Foreign Key)
* Count (Measure)
* Charge (Measure)

**Dimension Table 1: Time_Dim**
* Time_ID (Primary Key)
* Day
* Month
* Quarter
* Year

**Dimension Table 2: Doctor_Dim**
* Doctor_ID (Primary Key)
* Doctor_Name
* Specialty
* Hospital

**Dimension Table 3: Patient_Dim**
* Patient_ID (Primary Key)
* Patient_Name
* Age
* Address

### c) OLAP Operations
To list the total fee collected by each doctor in 2024 starting from the base cuboid

1.  **Roll-up** on the `time` dimension from `day` to `year`.
2.  **Dice** (or **Slice**) on the `time` dimension to select only `year = 2024`.
3.  **Roll-up** on the `patient` dimension from `patient` to `all` (effectively aggregating the data so individual patients are no longer separated).

### d) SQL Query
```sql
SELECT
    doctor,
    SUM(charge) AS total_fee
FROM
    fee
WHERE
    year = 2024
GROUP BY
    doctor;
```


__Problem 2__   (15pt)
Suppose that a data warehouse for Big University consists of the four dimensions student, course, semester, and instructor, and two measures count and avg grade. At the lowest conceptual level (e.g., for a given student, course, semester, and instructor combination), the avg grade measure stores the actual course grade of the student. At higher conceptual levels, avg grade stores the average grade for the given combination.

1. Draw a snowflake schema diagram for the data warehouse.
2. Starting with the base cuboid [student, course, semester, instructor], what specific OLAP operations (e.g., roll-up from semester to year) should you perform in order to list the average grade of CS courses for each Big University
3. If each dimension has five levels (including all), such as "student < major < status < university < all", how many cuboids will this cube contain. (including the base and apex cuboids)?

## ANSWER

### a) Snowflake Schema Diagram
In a Snowflake Schema, the dimension tables are normalized.

**Fact Table: Grade_Fact**
* Student_ID (Foreign Key)
* Course_ID (Foreign Key)
* Semester_ID (Foreign Key)
* Instructor_ID (Foreign Key)
* Count (Measure)
* Avg_Grade (Measure)

**Normalized Dimension Tables:**
* **Student_Dim:** Student_ID (PK), Name, Major_ID (FK)
* **Major_Dim:** Major_ID (PK), Major_Name, Department
* **Course_Dim:** Course_ID (PK), Course_Name, Department_ID (FK)
* **Department_Dim:** Department_ID (PK), Department_Name
* **Semester_Dim:** Semester_ID (PK), Term, Year
* **Instructor_Dim:** Instructor_ID (PK), Name, Title, Department_ID (FK)

### b) OLAP Operations
Assuming the prompt intends to ask for the average grade of CS courses for *each student*, you would perform the following operations starting from the base cuboid `[student, course, semester, instructor]`:
1.  **Roll-up** on the `course` dimension from `course` to `department`.
2.  **Dice** on the `course` dimension for `department = 'CS'`.
3.  **Roll-up** on the `semester` dimension from `semester` to `all`.
4.  **Roll-up** on the `instructor` dimension from `instructor` to `all`.

### c) Number of Cuboids
The total number of cuboids in a data cube is the product of the number of levels associated with each dimension. Let $L_i$ represent the number of levels in dimension $i$. The total number of cuboids $T$ is given by the formula:

$$T = \prod_{i=1}^{n} L_i$$

Given 4 dimensions (student, course, semester, instructor) and 5 levels each (including the "all" level):
Total Cuboids = $5 \times 5 \times 5 \times 5$ = **625 cuboids**.

Problem 3 (20pt) Suppose a company would like to design a data warehouse to facilitate the analysis of moving vehicles in an online analytical processing manner. The company registers huge amounts of auto movement data in the format of (Auto ID, location, speed, time). Each Auto ID represents a vehicle associated with information, such as vehicle category, driver category, etc., and each location may be associated with a street in a city. Assume that a street map is available for the city.

Design such a data warehouse to facilitate effective online analytical processing in multidimensional space.
The movement data may contain noise. Discuss how you would develop a method to automatically discover data records that were likely erroneously registered in the data repository.
The movement data may be sparse. Discuss how you would develop a method that constructs a reliable data warehouse despite the sparsity of data.
If one wants to drive from A to B starting at a particular time, discuss how a system may use the data in this warehouse to work out a fast route for the driver.

##Answer

### a) Data Warehouse Design


**Fact Table: Movement_Fact**
* Auto_ID (Foreign Key)
* Location_ID (Foreign Key)
* Time_ID (Foreign Key)
* Speed (Measure - typically averaged or maxed based on the query)
* Count (Measure - number of data points)

**Dimension Tables:**
* **Auto_Dim:** Auto_ID, Vehicle_Category, Driver_Category, Registration_State.
* **Location_Dim:** Location_ID, Street_Name, City, State, Latitude, Longitude.
* **Time_Dim:** Time_ID, Timestamp, Hour, Day_of_Week, Month, Year.

### b) Discovering Erroneous Records

* **Kinematic Constraints:** Filter out records where the recorded speed exceeds the physical capabilities of the vehicle or reasonable limits.
* **Derived Speed Validation:** Calculate the speed between two consecutive data points using the formula $v = \frac{\Delta d}{\Delta t}$ (where $\Delta d$ is distance and $\Delta t$ is time). If the derived speed severely contradicts the reported speed or physical limits, flag both points as potential noise.
* **Map Matching (Spatial Anomalies):** Compare the location coordinates against the provided street map. If a vehicle's coordinates place it in an impossible location (e.g., the middle of a lake or a building without roads), flag the record as a GPS error.

### c) Handling Data Sparsity
If movement data pings are sparse (e.g., recorded every 5 minutes instead of every 10 seconds), can build a reliable warehouse using:
* **Map-Matching and Routing Interpolation:** Use shortest-path algorithms on the street map to infer the likely streets taken between two distant location pings. can allocate fractional counts or estimated speeds to these inferred intermediate street segments.
* **Spatial and Temporal Aggregation:** Instead of storing data at the precise second or exact coordinate level, group the facts into larger bins. Aggregate the data by road segment rather than specific coordinates, and use 15-minute or 1-hour time blocks.

### d) Working Out Fast Route

* **Historical Average Query:** The system queries the warehouse for the average speed of all vehicles on specific street segments at the requested time of day and day of the week.
* **Cost Calculation:** It calculates the traversal cost (time) for each street segment using the formula $t = \frac{d}{v_{avg}}$, where $d$ is the segment distance and $v_{avg}$ is the historical average speed retrieved from the warehouse.
* **Pathfinding Algorithm:** Finally, it inputs these calculated traversal times as edge weights into a pathfinding algorithm (like Dijkstra's or A*) to calculate the route with the lowest total time cost from point A to point B.